# 04 · Teacher Candidate — ResNet18 (pretrained ImageNet)

ResNet18 fine-tuned from ImageNet weights using three-phase progressive unfreezing.
**This is the expected winner and selected teacher for knowledge distillation.**

**Justification for selection:**
- 11.2M parameters: sufficient capacity gap over the ~2.5M student (≈4.5× ratio)
- Residual connections generalize better than VGG on small datasets
- Pretrained features provide rich initialization: avoids overfitting on 7,000 images
- Faster convergence than VGG16-BN with less risk of instability

**Three-phase protocol:**
- Phase 1 (epochs 1–9):  backbone frozen, head only at lr=3e-4
- Phase 2 (epochs 10–19): unfreeze layer4 at lr=1e-4
- Phase 3 (epochs 20–30): unfreeze full backbone at lr=3e-5


In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


In [ ]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import (
    VGG_Scratch, VGG_Pretrained,
    ResNet_Scratch, ResNet18_Pretrained,
    MobileNetV2_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, MODEL_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_model, train_model_three_phase,
    train_multi_seed, train_kd, plot_history,
)

device = setup_device(seed=41)


In [ ]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


## Standardized hyperparameters

All models use identical settings to ensure fair comparison:

| Parameter | Scratch models | Pretrained models |
|-----------|---------------|-------------------|
| Batch size | 64 | 64 |
| Optimizer | Adam | Adam |
| Weight decay | 1e-4 | 1e-4 |
| Label smoothing | 0.1 | 0.1 |
| Augmentation | standard | standard |
| Scheduler | CosineAnnealingLR | CosineAnnealingLR |
| Patience | 10 | 10 |
| Seeds | [41, 52, 63] | [41, 52, 63] |
| Max epochs | 50 | 30 |
| LR | 1e-3 | 3e-4 → 1e-4 → 3e-5 (3-phase) |

Pretrained models use fewer max epochs because transfer learning converges faster.
The three-phase progressive unfreeze is only applicable to pretrained models.


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")


In [ ]:
results, best = train_multi_seed(
    model_fn     = ResNet18_Pretrained,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    seeds        = [41, 52, 63],
    save_dir     = SAVE_DIR,
    name_prefix  = "resnet18_pretrained",
    pretrained   = True,
    epochs          = 30,
    lr_phase1       = 3e-4,
    lr_phase2       = 1e-4,
    lr_phase3       = 3e-5,
    phase2_epoch    = 10,
    phase3_epoch    = 20,
    weight_decay    = 1e-4,
    label_smoothing = 0.1,
    patience        = 10,
)


In [ ]:
plot_history(best, title=f"ResNet18 Pretrained (seed {best['seed']})")

accs = [r["best_acc"] for r in results]
print(f"\nResNet18 Pretrained  |  Mean: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%  |  "
      f"Best: {best['best_acc']*100:.2f}% (seed {best['seed']})")
print(f"Best checkpoint: {best['save_path']}")
